# Calculate treatment and outcome

Store all of the causal effects between variables in one grid dataframe.

## Code setup

In [1]:
import numpy as np
import pandas as pd

## Generate some data

This is nonsense!

In [2]:
df_data = pd.DataFrame()
n_patients = 100

for col in ['onset_during_sleep', 'precise_onset_known', 'atrial_fibrillation', 'afib_anticoagulant']:
    df_data[col] = np.random.binomial(1, 0.5, n_patients)
df_data['stroke_team'] = np.random.choice(range(110), n_patients)
df_data['stroke_severity'] = np.random.choice(range(42), n_patients)
df_data['prior_disability'] = np.random.choice(range(6), n_patients)
df_data['age'] = np.random.choice(np.arange(37.5,95,5), n_patients)
df_data['arrival_to_scan_time'] = np.random.choice(np.arange(0,240,15), n_patients)

In [3]:
df_data.head()

,onset_during_sleep,precise_onset_known,atrial_fibrillation,afib_anticoagulant,stroke_team,stroke_severity,prior_disability,age,arrival_to_scan_time
0,0,1,1,0,104,12,1,87.5,105
1,0,1,1,0,99,33,5,72.5,90
2,0,1,0,1,82,7,2,67.5,150
3,1,1,0,0,14,26,4,37.5,105
4,0,0,1,0,28,12,1,92.5,195


Replace the single categorical stroke team column with a series of one-hot-encoded columns:

In [4]:
# Make new column for each stroke team:
df_data = pd.concat((df_data, pd.get_dummies(df_data['stroke_team']).astype(int)), axis='columns')
# Remove the original stroke team column:
df_data = df_data.drop('stroke_team', axis='columns')

In [5]:
original_features = list(df_data.columns)
features = original_features + ['thrombolysis', 'discharge_disability']

In [6]:
features

['onset_during_sleep',
 'precise_onset_known',
 'atrial_fibrillation',
 'afib_anticoagulant',
 'stroke_severity',
 'prior_disability',
 'age',
 'arrival_to_scan_time',
 2,
 3,
 5,
 6,
 7,
 8,
 9,
 11,
 12,
 13,
 14,
 18,
 20,
 21,
 24,
 25,
 28,
 29,
 31,
 32,
 33,
 34,
 36,
 38,
 39,
 40,
 42,
 43,
 45,
 47,
 48,
 49,
 51,
 55,
 56,
 58,
 60,
 61,
 62,
 65,
 66,
 68,
 70,
 71,
 72,
 73,
 74,
 76,
 77,
 78,
 80,
 82,
 83,
 85,
 87,
 93,
 94,
 95,
 98,
 99,
 100,
 103,
 104,
 106,
 107,
 108,
 'thrombolysis',
 'discharge_disability']

## Make empty effect dataframe

Make an empty dataframe with row and column names for all of our features plus the treatment and outcome:

In [39]:
df_effects = pd.DataFrame(columns=features, index=features, dtype=float)
# Fill missing data with zeros:
df_effects = df_effects.fillna(0.0)

The dataframe should have all of the correct column and row names, and all of the values should be zero:

In [40]:
df_effects.head()

,onset_during_sleep,precise_onset_known,atrial_fibrillation,afib_anticoagulant,stroke_severity,prior_disability,age,arrival_to_scan_time,0,3,...,98,99,101,103,104,106,107,109,thrombolysis,discharge_disability
onset_during_sleep,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
precise_onset_known,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
atrial_fibrillation,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
afib_anticoagulant,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
stroke_severity,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## Update some values

Make the causal effect go from each row name to each column name.

For example, to update the effect of onset during sleep on precise onset known:

In [41]:
df_effects.loc['onset_during_sleep', 'precise_onset_known'] = 0.1

And the effect of precise onset known on thrombolysis:

In [42]:
df_effects.loc['precise_onset_known', 'thrombolysis'] = 0.1

Check that this updated in the dataframe:

In [43]:
df_effects.head()

,onset_during_sleep,precise_onset_known,atrial_fibrillation,afib_anticoagulant,stroke_severity,prior_disability,age,arrival_to_scan_time,0,3,...,98,99,101,103,104,106,107,109,thrombolysis,discharge_disability
onset_during_sleep,0.0,0.1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
precise_onset_known,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.1,0.0
atrial_fibrillation,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
afib_anticoagulant,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
stroke_severity,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## Calculate treatment values

First generate some random noise:

In [44]:
noise_t = np.random.normal(0, 0.1, n_patients)

Use a linear combination of this noise and the features to calculate a value for treatment for each patient.

In [45]:
# Calculate data by combining these generated data and causal effects.
# Treatment:
t = np.sum((
    [noise_t] +
    [df_data[col] * df_effects.loc[col, 'thrombolysis'] for col in df_data.columns]
), axis=0)

Check the first few treatment values:

In [46]:
t[:5]

array([-0.02723643,  0.20647491,  0.18436356, -0.02416402,  0.12659574])

## Convert treatment values to binary

Use a conversion function to decide whether each person is treated based on their score. People with higher scores are more likely (but not guaranteed) to be treated.

Use these functions from the DoWhy package:

In [47]:
# Functions for converting treatment calculation to binary decision.

import math

# Functions from dowhy `datasets` module:
# https://www.pywhy.org/dowhy/v0.14/_modules/dowhy/datasets.html#linear_dataset

def sigmoid(x):
    return 1 / (1 + math.exp(-x))

def convert_to_binary(x, stochastic=True):
    p = sigmoid(x)
    if stochastic:
        return np.random.choice([0, 1], 1, p=[1 - p, p])
    else:
        return int(p > 0.5)

Run the conversion function:

In [48]:
# Convert treatment to binary:
t_bin = np.array([convert_to_binary(x) for x in t]).flatten()

Check the first few results:

In [49]:
t_bin[:5]

array([1, 0, 1, 0, 0])

Put this with the rest of the data:

In [50]:
df_data['thrombolysis'] = t_bin